# Train DustyLM: Dusty in a Few Commands

This notebook shows the simple path to training Dusty, a tiny ~8M parameter assistant that talks like a small vacuum robot.

**What you will do:**
1. Check your environment
2. Train the tokenizer
3. Prepare tokenized data
4. Train the base Dusty model
5. Fine-tune it for chat
6. Try the result

The project already has the model, data prep, and training loop wired up. The notebook is mostly a friendly wrapper around `make` commands.

## Quick Start

Run these cells from the repository root. For a first smoke test, keep `EPOCHS=1`. For a better Dusty, increase the epoch counts later.

If you do not already have the training files, start with `make download-datasets`. It downloads TinyStories for pretraining and the Dusty SFT JSONL from the Hugging Face repo `mkhordoo/dusty-chat`.


In [ ]:
# Install the required dependencies
!pip install -q uv huggingface_hub && v pip install --system -e .

from pathlib import Path
import sys
import torch

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "dustylm").exists():
    sys.path.insert(0, str(REPO_ROOT))

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("Repo root:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("Training device:", device)


### Optional: Download the Datasets

If you already have `artifacts/datasets/dusty_pretrain.txt` and `artifacts/datasets/dusty_sft.jsonl`, skip this cell. Otherwise, this is the one-command path to get the training data.


In [ ]:
!make download-datasets


### 1. Check the Training Text

Training needs Dusty text under `artifacts/datasets/`. If you already generated or downloaded the data, this cell will show green checks.

If files are missing, use the optional data-generation commands in the Advanced section, or place your own Dusty-style text at the same paths.

In [ ]:
required_sources = {
    "pretrain text": REPO_ROOT / "artifacts" / "datasets" / "dusty_pretrain.txt",
    "SFT chat JSONL": REPO_ROOT / "artifacts" / "datasets" / "dusty_sft.jsonl",
}

for label, path in required_sources.items():
    status = "found" if path.exists() else "missing"
    print(f"{label:16} {status:7} {path}")


### 2. Train the Tokenizer

Dusty uses a small 4096-token BPE vocabulary. This command trains the tokenizer from the local Dusty text and writes `artifacts/tokenizers/dusty_tokenizer.json`.

In [ ]:
!make dusty-tokenizer


### 3. Prepare Tokenized Datasets

These commands turn raw text and chat JSONL into Hugging Face `datasets` saved on disk. The training loop reads these tokenized datasets directly.

In [ ]:
!make dusty-pretrain-data
!make dusty-sft-data


### 4. Train Dusty

The base model learns language from plain text. Then SFT teaches it to answer user messages in ChatML format.

**Note:** `EPOCHS=1` is a smoke test. It verifies that the pipeline works, but Dusty may output gibberish. To get the coherent vacuum persona shown in the README, train the base model to roughly **160M tokens** (about **7 epochs** with the default data size) and then SFT for **1-2 epochs**.


In [ ]:
!make dusty-pretrain EPOCHS=1


In [ ]:
!make dusty-sft-train EPOCHS=1


### 5. Try the Trained Model

After SFT, the checkpoint should be at `artifacts/checkpoints/dusty8m_sft.pt`. Load it with the same inference API used in the quick-start notebook.

In [ ]:
from dustylm.inference import Inference

engine = Inference(
    checkpoint_path=REPO_ROOT / "artifacts" / "checkpoints" / "dusty8m_sft.pt",
    tokenizer_path=REPO_ROOT / "artifacts" / "tokenizers" / "dusty_tokenizer.json",
    device=device,
)


def chat(prompt):
    response = engine.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=64,
        temperature=0.6,
        top_p=0.8,
    )
    return response["choices"][0]["message"]["content"].strip()


for prompt in ["who are you?", "what is under the sofa?", "are you afraid of the cat?"]:
    print(f"You> {prompt}")
    print(f"Dusty> {chat(prompt)}\n")


## Advanced: Generate the Training Data

If you do not already have Dusty datasets, this repo can generate synthetic data. These steps may call an external model provider, so set your API key first.

The key point: once data exists, training is just the small set of `make` commands above.

In [ ]:
# Optional: generate pretraining text and SFT chat examples.
# Uncomment these when your model/API credentials are configured.
# !make dusty-generate-pretrain
# !make dusty-generate-sft


### Changing Dusty's Persona

If you simply want to retrain Dusty, use the downloaded dataset from `make download-datasets` and keep the training flow unchanged.

If you want a different robot persona, create a new SFT dataset by editing the prompt inside `dataset_generation/generate_sft_dataset_with_fallback.py`. Change the instructions that define Dusty's personality, topics, and speaking style, then run `make dusty-generate-sft` to produce your new chat JSONL before training.


## Useful Commands

Run these in a terminal when you do not need the notebook:

```bash
make download-datasets
make dusty-tokenizer
make dusty-pretrain-data
make dusty-pretrain EPOCHS=1
make dusty-sft-data
make dusty-sft-train EPOCHS=1
make chat
```

That is the whole loop: download data → tokenizer → pretrain → SFT → chat.
